# YOLO + SAM: Detectando y Segmentando el Mundo 🌍

Bienvenidos a la demostración interactiva de visión computacional. En esta sesión no solo veremos cómo usar modelos estelares como **YOLO** (You Only Look Once) o **SAM** (Segment Anything Model), sino que entenderemos *por qué* necesitan trabajar juntos.

### 🖥️ Preparando el escenario 

In [1]:
import torch

# YOLO11 es tan versátil que correrá en CPU, pero si tienes una GPU (NVIDIA CUDA) o Mac M1/M2 (MPS), la usaremos.
device = "cuda" if torch.cuda.is_available() else \
         "mps" if torch.backends.mps.is_available() else "cpu"

print(f"🔥 Dispositivo seleccionado para inferencia: {device}")

🔥 Dispositivo seleccionado para inferencia: cuda


## 🎭 Acto 1: SAM y el Asombro del Detalle
Empecemos con **FastSAM**, una versión ultra-rápida de Segment Anything Model. 
SAM es capaz de recortar **cualquier** objeto en una imagen con precisión de píxel a píxel. Es ciego en cuanto a significado (no sabe qué es un 'perro' o una 'persona'), pero mecánicamente es un genio del recorte.

In [2]:
from ultralytics import FastSAM

# Cargamos FastSAM (versión pequeña para maximizar velocidad)
sam = FastSAM("FastSAM-s.pt")

# Le pedimos al modelo que fragmente ABSOLUTAMENTE TODO en la imagen
results_sam = sam("assets/sample_images/office.jpg", device=device, retina_masks=True, conf=0.4)

# Mostrar el resultado interactivo con los recortes
results_sam[0].show()


image 1/1 c:\Users\efrai\OneDrive\Desktop\EF\Education\Professor\DL classes\uag-deep-learning\parcial_3\yolo\assets\sample_images\office.jpg: 384x640 17 objects, 493.9ms
Speed: 12.5ms preprocess, 493.9ms inference, 216.1ms postprocess per image at shape (1, 3, 384, 640)


In [3]:
results_sam[0].show()

## 🎭 Acto 2: YOLO y el Poder del Contexto
A diferencia de SAM, **YOLO** sabe perfectamente qué está viendo (si la clase está dentro de sus 80 categorías conocidas). Sacrifica los bordes curvados perfectos a cambio de **velocidad extrema** y cajas delimitadoras que te dicen: *"Ahí hay un auto, acá una persona"*.

In [4]:
from ultralytics import YOLO

# Cargamos el modelo nano de YOLO11 (6 MB)
model = YOLO("yolo11n.pt")

# Hacemos inferencia sobre una imagen urbana
results_yolo = model("assets/sample_images/street.jpg", device=device)
results_yolo[0].show()

# Podemos inspeccionar el contenido que detectó:
print(f"\n🔎 YOLO detectó {len(results_yolo[0].boxes)} objetos reconocidos.")
print(f"Nivel de confianza de la primera detección: {results_yolo[0].boxes.conf[0]:.2f}")


image 1/1 c:\Users\efrai\OneDrive\Desktop\EF\Education\Professor\DL classes\uag-deep-learning\parcial_3\yolo\assets\sample_images\street.jpg: 640x480 4 persons, 1 bus, 148.5ms
Speed: 11.1ms preprocess, 148.5ms inference, 2.9ms postprocess per image at shape (1, 3, 640, 480)

🔎 YOLO detectó 5 objetos reconocidos.
Nivel de confianza de la primera detección: 0.94


### Cuando YOLO brilla: Video 📽️
Correr cajas en tiempo real es el verdadero de superpoder de YOLO. Vamos a procesar un video completo a decenas de cuadros por segundo.

In [5]:
# Esto guardará un nuevo .mp4 en la carpeta 'runs/detect/predict/' con todas las detecciones incrustadas
results_video = model.predict(
    source="assets/video_demo.mp4", 
    save=True, 
    conf=0.4, 
    device=device
)
print("✅ ¡Video procesado exitosamente! Revisa la carpeta runs/detect/")


WARNING 
Inference results will accumulate in RAM unless `stream=True` is passed, which can cause out-of-memory errors for large
sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict/ for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs

video 1/1 (frame 1/1116) c:\Users\efrai\OneDrive\Desktop\EF\Education\Professor\DL classes\uag-deep-learning\parcial_3\yolo\assets\video_demo.mp4: 288x640 (no detections), 134.9ms
video 1/1 (frame 2/1116) c:\Users\efrai\OneDrive\Desktop\EF\Education\Professor\DL classes\uag-deep-learning\parcial_3\yolo\assets\video_demo.mp4: 288x640 (no detections), 27.0ms
video 1/1 (frame 3/1116) c:\Users\efrai\OneDrive\Desktop\EF\Education\Professor\DL classes\uag-deep-le

## 🎭 Acto 3: El Matrimonio Perfecto (Pipeline YOLO 🤝 SAM)
¿Qué pasa si queremos lo mejor de ambos mundos? 
- Queremos saber **QUÉ** son los objetos y filtrarlos velozmente (YOLO).
- Queremos **RECORTARLOS** con máscaras perfectas para producción gráfica (SAM).

> **Estrategia (Detector-then-Segmenter)**: Usamos las cajas que escupe YOLO como las *pistas (prompts)* para decirle a SAM exactamente *dónde* tiene que hacer sus máscaras perfectas.

In [8]:
# 1. YOLO detecta todo en la imagen de construcción
det_results = model("assets/sample_images/construction.jpg", device=device)

# 2. Extraemos ÚNICAMENTE las coordenadas de las cajas (nuestros Prompts)
boxes = det_results[0].boxes.xyxy.tolist()
print(f"Se encontraron {len(boxes)} elementos de interés.")

# 3. Alimentamos esas cajas directamente a SAM
sam_results = sam("assets/sample_images/construction.jpg", bboxes=boxes, device=device)
sam_results[0].show()


image 1/1 c:\Users\efrai\OneDrive\Desktop\EF\Education\Professor\DL classes\uag-deep-learning\parcial_3\yolo\assets\sample_images\construction.jpg: 448x640 4 persons, 114.5ms
Speed: 3.5ms preprocess, 114.5ms inference, 4.7ms postprocess per image at shape (1, 3, 448, 640)
Se encontraron 4 elementos de interés.

image 1/1 c:\Users\efrai\OneDrive\Desktop\EF\Education\Professor\DL classes\uag-deep-learning\parcial_3\yolo\assets\sample_images\construction.jpg: 448x640 4 objects, 144.6ms
Speed: 4.4ms preprocess, 144.6ms inference, 245.7ms postprocess per image at shape (1, 3, 448, 640)


## 🎭 Acto 4: Crea tu propio YOLO (Fine-Tuning) 🛠️
YOLO11 fue entrenado con el dataset COCO (80 clases comunes), pero si tienes un problema de negocio (por ejemplo, detectar **Cascos de Seguridad** en una obra civil para verificar compliance), puedes seguir aprovechando su poderosa estructura base y simplemente cambiarle las clases finales.

Para esto, basta un comando y un archivo `data.yaml`.

In [10]:
# Partimos de los pesos pre-entrenados para no empezar de cero (Transfer Learning)
model_ft = YOLO("yolo11n.pt")

# Y ejecutamos el ciclo de entrenamiento
# Mientras esto corre, presten atención adelante 👀
model_ft.train(
    data="dataset/data.yaml",  # El archivo que le dice dónde están las imágenes de entrenamiento
    epochs=10,                 # Un fine-tuning ultra veloz
    imgsz=320,                 # Reducimos resolución para acelerar CPU
    batch=8,
    device=device,
    plots=True                 # ¡Generará gráficas de su progreso automáticamente!
)

Ultralytics 8.4.33  Python-3.10.6 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 3070 Laptop GPU, 8192MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=dataset/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=10, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=320, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train2, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, pers

RuntimeError: Dataset 'dataset/data.yaml' error  Dataset 'dataset/data.yaml' images not found, missing path 'C:\Users\efrai\OneDrive\Desktop\EF\Education\Professor\DL classes\datasets\hardhat\coco128\images\train2017'
Note dataset download directory is 'C:\Users\efrai\OneDrive\Desktop\EF\Education\Professor\DL classes\datasets'. You can update this in 'C:\Users\efrai\AppData\Roaming\Ultralytics\settings.json'

## 🎭 Acto 5: Tu Modelo Listo para el Mundo Real 📱
Una vez que el modelo se ha entrenado, probarlo es tan fácil como cuando instanciamos YOLO la primera vez.
Además, puedes exportarlo en formato **ONNX** para poder desplegarlo en C++, en la web o en aplicaciones móviles pesando menos de 10 MB.

In [ ]:
# Ultralytics guarda tu mejor modelo automáticamente en la carpeta runs/detect/train/weights/
try:
    my_new_model = YOLO("runs/detect/train/weights/best.pt")
    
    # ¡Pruébalo con una nueva imagen de construcción que no haya visto!
    results = my_new_model("assets/sample_images/construction.jpg", device=device)
    results[0].show()

    # Exportando el modelo para uso en producción
    print("\n📦 Exportando modelo...")
    my_new_model.export(format="onnx")
    print("\n🚀 ¡Tu YOLO ahora vive fuera de Python y puede ser usado en cualquier framework o teléfono!")
except Exception as e:
    print(f"Asegúrate de que el entrenamiento haya terminado. Error: {e}")